# SQLite Manual Analysis

Use this notebook to inspect the SQLite database created by the Rekordbox ingestion pipeline.

In [1]:
from pathlib import Path
import os
import sqlite3

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if load_dotenv:
    load_dotenv(PROJECT_ROOT / ".env")

db_path = Path(os.getenv("SQLITE_DB_PATH", "data/rekordbox_tracks.sqlite3"))
if not db_path.is_absolute():
    db_path = PROJECT_ROOT / db_path

print(f"Project root: {PROJECT_ROOT}")
print(f"SQLite database: {db_path}")
print(f"Database exists: {db_path.exists()}")

Project root: /Users/zacurbiztondo/dj-library-helper
SQLite database: /Users/zacurbiztondo/dj-library-helper/data/rekordbox_tracks.sqlite3
Database exists: True


## List Tables

In [2]:
if not db_path.exists():
    raise FileNotFoundError(f"SQLite database not found: {db_path}")

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

tables = conn.execute(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """
).fetchall()

table_names = [row["name"] for row in tables]
print("Tables:")
for table_name in table_names:
    print(f"- {table_name}")

table_names

Tables:
- genres
- harmony
- rekordbox_spotify_matches
- rekordbox_tracks
- rhythm
- score
- spotify_tracks
- track_analysis


['genres',
 'harmony',
 'rekordbox_spotify_matches',
 'rekordbox_tracks',
 'rhythm',
 'score',
 'spotify_tracks',
 'track_analysis']

## Print Schemas And Row Counts

In [3]:
for table_name in table_names:
    print(f"\n=== {table_name} ===")
    row_count = conn.execute(f'SELECT COUNT(*) AS count FROM "{table_name}"').fetchone()["count"]
    print(f"Rows: {row_count}")
    print("Columns:")
    for column in conn.execute(f'PRAGMA table_info("{table_name}")').fetchall():
        print(
            f"- {column['name']} {column['type']} "
            f"nullable={not bool(column['notnull'])} pk={bool(column['pk'])}"
        )


=== genres ===
Rows: 209
Columns:
- rekordbox_track_id INTEGER nullable=False pk=True
- spotify_track_id VARCHAR(64) nullable=False pk=False
- values TEXT nullable=True pk=False
- raw_payload TEXT nullable=True pk=False
- fetched_at DATETIME nullable=False pk=False

=== harmony ===
Rows: 209
Columns:
- rekordbox_track_id INTEGER nullable=False pk=True
- spotify_track_id VARCHAR(64) nullable=False pk=False
- key INTEGER nullable=True pk=False
- mode VARCHAR(64) nullable=True pk=False
- camelot VARCHAR(16) nullable=True pk=False
- camelot_number INTEGER nullable=True pk=False
- camelot_letter VARCHAR(1) nullable=True pk=False
- note VARCHAR(16) nullable=True pk=False
- raw_payload TEXT nullable=True pk=False
- fetched_at DATETIME nullable=False pk=False

=== rekordbox_spotify_matches ===
Rows: 209
Columns:
- rekordbox_track_id INTEGER nullable=False pk=True
- spotify_track_id VARCHAR(64) nullable=False pk=False
- match_score FLOAT nullable=False pk=False
- spotify_search_query_string TE

## Load Rekordbox Tracks

In [4]:
try:
    import pandas as pd
except ImportError as exc:
    raise ImportError("Install pandas to use the analysis cells: pip install pandas") from exc

tracks = pd.read_sql_query("SELECT * FROM rekordbox_tracks", conn)
print(f"Loaded {len(tracks)} tracks")
tracks.head(20)

Loaded 289 tracks


,rekordbox_track_id,title,artist,album,genre,bpm,key,rating,comments,duration,date_added,file_path,playlist_name,spotify_search_query_string
0,257614,baby again.. (original mix),"four tet, skrillex, fred again..",Baby again..,Bass House,127.00,Gm,0,,319.0,2024-11-09 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/Baby agai...,Z,track: baby again artist: four tet album: Baby...
1,549049,go dj [explicit],lil wayne,Tha Carter [Explicit],Rap & Hip-Hop,79.00,Em,0,Amazon.com Song ID: 202366200,281.0,2024-11-09 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/Rap/Go DJ...,Z,track: go dj artist: lil wayne album: Tha Cart...
2,786655,murdah (isoxo remix),knock2,niteharts,Trap / Future Bass,145.00,Dm,0,,172.0,2026-03-31 00:00:00.000000,Users/zacurbiztondo/Downloads/beatport_tracks_...,Z,track: murdah (isoxo remix) artist: knock2 alb...
3,1112161,on my mind (original mix),"diplo, sidepiece",On My Mind,Bass House,123.00,Gm,0,,189.0,2024-11-09 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/On My Min...,Z,track: on my mind artist: diplo album: On My Mind
4,1500026,tears,"skrillex, joker, & sleepnet",,,140.00,Fm,0,https://www.youtube.com/watch?,185.0,2026-05-14 00:00:00.000000,"Users/zacurbiztondo/Desktop/DJ-Music/Skrillex,...",Z,track: tears artist: skrillex album:
5,1516396,"madonna,the weeknd, playboi carti - popular [w...",NaN,,,99.00,Bbm,0,,216.0,2024-11-09 00:00:00.000000,"Users/zacurbiztondo/Desktop/DJ-Music/Madonna,T...",Z,track: madonna the weeknd playboi carti popula...
6,2967406,just jammin' (original mix),gramatik,Coffee Shop Selection,Electronica,90.00,Ebm,0,,389.0,2025-11-28 00:00:00.000000,Users/zacurbiztondo/Downloads/beatport_tracks_...,Z,track: just jammin artist: gramatik album: Cof...
7,7208611,i don't like,chief keef,Still Rich,Drill Rap,131.99,F#m,0,,294.0,2026-05-14 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/I Don't L...,Z,track: i don t like artist: chief keef album: ...
8,8784213,popular (sped up),NaN,,,114.84,Abm,0,,187.0,2024-11-09 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/Popular (...,Z,track: popular album:
9,9901726,skank n flex (original mix),"wax motif, scrufizzer, taiki nulight",House of Wax,Bass House,130.00,Fm,0,,266.0,2025-01-07 00:00:00.000000,Users/zacurbiztondo/Desktop/DJ-Music/Wax Motif...,Z,track: skank n flex artist: wax motif album: H...


## Load Spotify Enrichment Tables

Load the normalized Spotify enrichment output tables when they exist. This keeps the notebook usable before and after running `scripts/enrich_spotify.py`.

In [5]:
def table_exists(table_name):
    return table_name in table_names


def table_columns(table_name):
    if not table_exists(table_name):
        return set()
    return {
        row["name"]
        for row in conn.execute(f'PRAGMA table_info("{table_name}")').fetchall()
    }

if table_exists("spotify_tracks"):
    spotify_tracks = pd.read_sql_query("SELECT * FROM spotify_tracks", conn)
else:
    spotify_tracks = pd.DataFrame()

if table_exists("rekordbox_spotify_matches"):
    rekordbox_spotify_matches = pd.read_sql_query(
        "SELECT * FROM rekordbox_spotify_matches",
        conn,
    )
else:
    rekordbox_spotify_matches = pd.DataFrame()

print(f"Loaded {len(spotify_tracks)} Spotify tracks")
print(f"Loaded {len(rekordbox_spotify_matches)} Rekordbox/Spotify matches")

if not spotify_tracks.empty:
    display(spotify_tracks.head(20))

if not rekordbox_spotify_matches.empty:
    display(rekordbox_spotify_matches.head(20))

Loaded 276 Spotify tracks
Loaded 209 Rekordbox/Spotify matches


,spotify_track_id,title,artist_names,album_name,duration_ms,explicit,spotify_url,spotify_uri,fetched_at
0,4SxLCXoMTrQG1VeOhulg6h,Baby Again..,"Fred again.., Skrillex, Four Tet",Baby Again - Electro Pop Hits,319963,0,https://open.spotify.com/track/4SxLCXoMTrQG1Ve...,spotify:track:4SxLCXoMTrQG1VeOhulg6h,2026-06-09 22:06:24
1,1O1KXvUMM5VRvgMDGBmU5U,Just Jammin' NYC,"Gramatik, Exmag",Coffee Shop Selection,367058,0,https://open.spotify.com/track/1O1KXvUMM5VRvgM...,spotify:track:1O1KXvUMM5VRvgMDGBmU5U,2026-06-09 22:06:28
2,3sh2q76qsc7yLkQNmHKfQf,Stephanie,"Cloonee, Young M.A, InntRaw",Stephanie,188534,0,https://open.spotify.com/track/3sh2q76qsc7yLkQ...,spotify:track:3sh2q76qsc7yLkQNmHKfQf,2026-06-13 05:15:20
3,4T6GfkSM77XCgPIGbaNtkq,dashstar* - VIP,Knock2,dashstar* (Yellow Claw Trap Edit),140481,0,https://open.spotify.com/track/4T6GfkSM77XCgPI...,spotify:track:4T6GfkSM77XCgPIGbaNtkq,2026-06-09 22:06:35
4,2GFAanJTKSuI4uFfrypQwc,Think About Us,"Sonny Fodera, D.O.D, Poppy Baskcomb",Think About Us (feat. Poppy Baskcomb) [Extende...,178148,0,https://open.spotify.com/track/2GFAanJTKSuI4uF...,spotify:track:2GFAanJTKSuI4uFfrypQwc,2026-06-09 22:06:38
5,18ty2CvaOkyz3WOD3DkqNk,CALYPSO,"ero808, NXSTY",CALYPSO,152015,1,https://open.spotify.com/track/18ty2CvaOkyz3WO...,spotify:track:18ty2CvaOkyz3WOD3DkqNk,2026-06-09 22:06:40
6,6mYZtbF8kIWAFuu0xFUHxH,SKYFALLING,"Wavedash, Flux Pavilion, meesh",Tempo,178666,0,https://open.spotify.com/track/6mYZtbF8kIWAFuu...,spotify:track:6mYZtbF8kIWAFuu0xFUHxH,2026-06-09 22:06:43
7,1DtxaBTTZKfRRlOBiPgpNI,Are U Feelin Me,"Knock2, DEV",ROOM202 dlux,184875,0,https://open.spotify.com/track/1DtxaBTTZKfRRlO...,spotify:track:1DtxaBTTZKfRRlOBiPgpNI,2026-06-13 04:37:33
8,5Zg8ogzHlYTAU0IFy8TZh1,4EVR,"ISOKNOCK, ISOxo, Knock2, cade clair",4EVR,195428,0,https://open.spotify.com/track/5Zg8ogzHlYTAU0I...,spotify:track:5Zg8ogzHlYTAU0IFy8TZh1,2026-06-09 22:06:47
9,2tkgvVIpAMPLHKa2NLQbkH,What's the Move (feat. General Degree) - RemK ...,"Henry Fong, Knock2, General Degree, RemK",What's the Move (feat. General Degree) [Remixes],193794,0,https://open.spotify.com/track/2tkgvVIpAMPLHKa...,spotify:track:2tkgvVIpAMPLHKa2NLQbkH,2026-06-09 22:06:51


,rekordbox_track_id,spotify_track_id,match_score,spotify_search_query_string,matched_at
0,257614,4zlbKky2yA657Sk5rekZoR,0.8333,track: baby again artist: four tet album: Baby...,2026-06-13 04:34:56
1,549049,2x2iWUo1PYncvoUQO8M94J,0.9690,track: go dj artist: lil wayne album: Tha Cart...,2026-06-13 05:15:06
2,786655,1wecjDqjR28ENc6G4oGIBh,0.9000,track: murdah (isoxo remix) artist: knock2 alb...,2026-06-13 05:15:08
3,1112161,54hA0ldJYyT1huGfSeOjdQ,0.7682,track: on my mind artist: diplo album: On My Mind,2026-06-13 04:34:59
4,1500026,29tRu5BqKiUDAoWLKIO3RN,0.8387,track: tears artist: skrillex album:,2026-06-13 05:15:09
5,2967406,0h7JPd0usccTo7EoCXiNck,0.9083,track: just jammin artist: gramatik album: Cof...,2026-06-13 05:15:11
6,7208611,1h6kgem1ai8vUgO1rZOwfB,0.8727,track: i don t like artist: chief keef album: ...,2026-06-13 05:15:13
7,10261590,3Fc7k96EGOGiJBMZUxbpq7,1.0000,track: lite spots artist: kaytranada album: 99...,2026-06-13 05:15:16
8,13594113,03tqyYWC9Um2ZqU0ZN849H,0.8769,track: no hands artist: waka flocka flame albu...,2026-06-13 05:15:19
9,14877728,3sh2q76qsc7yLkQNmHKfQf,0.8313,track: stephanie artist: cloonee album: Stephanie,2026-06-13 05:15:20


## Load DJ Audio Analysis Tables

Load the normalized DJ Track Audio Analysis enrichment tables when they exist. These are populated by `scripts/enrich_dj_audio_analysis.py`.

In [6]:
dj_audio_analysis_table_names = [
    "track_analysis",
    "rhythm",
    "harmony",
    "score",
    "genres",
]

dj_audio_analysis_tables = {}
for table_name in dj_audio_analysis_table_names:
    if table_exists(table_name):
        dj_audio_analysis_tables[table_name] = pd.read_sql_query(
            f'SELECT * FROM "{table_name}"',
            conn,
        )
    else:
        dj_audio_analysis_tables[table_name] = pd.DataFrame()

track_analysis = dj_audio_analysis_tables["track_analysis"]
rhythm = dj_audio_analysis_tables["rhythm"]
harmony = dj_audio_analysis_tables["harmony"]
score = dj_audio_analysis_tables["score"]
genres = dj_audio_analysis_tables["genres"]

for table_name, frame in dj_audio_analysis_tables.items():
    print(f"Loaded {len(frame)} rows from {table_name}")

for table_name, frame in dj_audio_analysis_tables.items():
    if not frame.empty:
        print(f"\n{table_name}")
        display(frame.head(20))


Loaded 209 rows from track_analysis
Loaded 209 rows from rhythm
Loaded 209 rows from harmony
Loaded 209 rows from score
Loaded 209 rows from genres

track_analysis


,rekordbox_track_id,spotify_track_id,ids_spotify,ids_isrc,href,name,popularity,duration,duration_s,duration_ms,loudness,loudness_db,is_vocal_heavy,is_acoustic,is_instrumental,is_live_recording,is_club_loud,raw_payload,fetched_at
0,257614,4zlbKky2yA657Sk5rekZoR,4zlbKky2yA657Sk5rekZoR,GBAHS2300215,https://api.spotify.com/v1/tracks/4zlbKky2yA65...,Baby again..,61,05:19,320.0,319964,-7.7 dB,-7.668,0,0,0,0,0,"{""duration"": ""05:19"", ""duration_ms"": 319964, ""...",2026-06-13 19:18:34
1,549049,2x2iWUo1PYncvoUQO8M94J,2x2iWUo1PYncvoUQO8M94J,USCM50400067,https://api.spotify.com/v1/tracks/2x2iWUo1PYnc...,Go DJ,12,04:41,281.8,281827,-3.1 dB,-3.137,0,0,0,0,1,"{""duration"": ""04:41"", ""duration_ms"": 281827, ""...",2026-06-13 19:18:34
2,786655,1wecjDqjR28ENc6G4oGIBh,1wecjDqjR28ENc6G4oGIBh,QMUY42200172,https://api.spotify.com/v1/tracks/1wecjDqjR28E...,murdah (ISOxo Remix),45,02:52,172.1,172138,-0.8 dB,-0.842,0,0,0,0,1,"{""duration"": ""02:52"", ""duration_ms"": 172138, ""...",2026-06-13 19:18:34
3,1112161,54hA0ldJYyT1huGfSeOjdQ,54hA0ldJYyT1huGfSeOjdQ,USZ4V1900134,https://api.spotify.com/v1/tracks/54hA0ldJYyT1...,On My Mind,64,03:09,189.3,189268,-6.6 dB,-6.577,0,0,1,0,1,"{""duration"": ""03:09"", ""duration_ms"": 189268, ""...",2026-06-13 19:18:34
4,1500026,29tRu5BqKiUDAoWLKIO3RN,29tRu5BqKiUDAoWLKIO3RN,USAT22300825,https://api.spotify.com/v1/tracks/29tRu5BqKiUD...,Tears,51,03:05,185.6,185571,-5.3 dB,-5.288,0,0,0,0,1,"{""duration"": ""03:05"", ""duration_ms"": 185571, ""...",2026-06-13 19:18:34
5,2967406,0h7JPd0usccTo7EoCXiNck,0h7JPd0usccTo7EoCXiNck,USQY51422103,https://api.spotify.com/v1/tracks/0h7JPd0usccT...,Just Jammin',53,06:29,389.5,389500,-5.8 dB,-5.785,0,0,0,0,1,"{""duration"": ""06:29"", ""duration_ms"": 389500, ""...",2026-06-13 19:18:35
6,7208611,1h6kgem1ai8vUgO1rZOwfB,1h6kgem1ai8vUgO1rZOwfB,USUM71207543,https://api.spotify.com/v1/tracks/1h6kgem1ai8v...,I Don't Like,74,04:53,293.8,293840,-4.6 dB,-4.622,0,0,0,0,1,"{""duration"": ""04:53"", ""duration_ms"": 293840, ""...",2026-06-13 19:18:35
7,10261590,3Fc7k96EGOGiJBMZUxbpq7,3Fc7k96EGOGiJBMZUxbpq7,GBBKS1600025,https://api.spotify.com/v1/tracks/3Fc7k96EGOGi...,LITE SPOTS,52,03:50,230.9,230920,-11.7 dB,-11.683,1,0,0,0,0,"{""duration"": ""03:50"", ""duration_ms"": 230920, ""...",2026-06-13 19:18:35
8,13594113,03tqyYWC9Um2ZqU0ZN849H,03tqyYWC9Um2ZqU0ZN849H,USWB11002103,https://api.spotify.com/v1/tracks/03tqyYWC9Um2...,No Hands (feat. Roscoe Dash & Wale),78,04:23,263.8,263773,-6.4 dB,-6.366,0,0,0,0,1,"{""duration"": ""04:23"", ""duration_ms"": 263773, ""...",2026-06-13 19:18:35
9,14877728,3sh2q76qsc7yLkQNmHKfQf,3sh2q76qsc7yLkQNmHKfQf,GBLV62409221,https://api.spotify.com/v1/tracks/3sh2q76qsc7y...,Stephanie,71,03:08,188.5,188534,-6.8 dB,-6.838,0,0,0,0,1,"{""duration"": ""03:08"", ""duration_ms"": 188534, ""...",2026-06-13 19:18:35



rhythm


,rekordbox_track_id,spotify_track_id,bpm,tempo,bucket,beats,beats_per_bar,beat_duration_ms,bars,time_signature,...,phrases_s_bar_2,phrases_s_bar_4,phrases_s_bar_8,phrases_s_bar_16,phrases_s_bar_32,phrases_s_bar_64,phrases_count_bar_16,phrases_count_bar_32,raw_payload,fetched_at
0,257614,4zlbKky2yA657Sk5rekZoR,126.983,127.00 BPM,allegro,677,4,473,169,4/4,...,3.8,7.6,15.1,30.2,60.5,121.0,10,5,"{""bars"": 169, ""beat_duration_ms"": 473, ""beats""...",2026-06-13 19:18:34
1,549049,2x2iWUo1PYncvoUQO8M94J,157.955,158.00 BPM,allegro,742,4,380,185,4/4,...,3.0,6.1,12.2,24.3,48.6,97.2,11,5,"{""bars"": 185, ""beat_duration_ms"": 380, ""beats""...",2026-06-13 19:18:34
2,786655,1wecjDqjR28ENc6G4oGIBh,144.937,145.00 BPM,allegro,416,4,414,104,4/4,...,3.3,6.6,13.2,26.5,53.0,106.0,6,3,"{""bars"": 104, ""beat_duration_ms"": 414, ""beats""...",2026-06-13 19:18:34
3,1112161,54hA0ldJYyT1huGfSeOjdQ,123.027,123.00 BPM,allegro,388,4,488,97,4/4,...,3.9,7.8,15.6,31.2,62.4,124.9,6,3,"{""bars"": 97, ""beat_duration_ms"": 488, ""beats"":...",2026-06-13 19:18:34
4,1500026,29tRu5BqKiUDAoWLKIO3RN,140.216,140.00 BPM,allegro,434,4,428,108,4/4,...,3.4,6.8,13.7,27.4,54.8,109.5,6,3,"{""bars"": 108, ""beat_duration_ms"": 428, ""beats""...",2026-06-13 19:18:34
5,2967406,0h7JPd0usccTo7EoCXiNck,90.008,90.00 BPM,andante,584,4,667,146,4/4,...,5.3,10.7,21.3,42.7,85.3,170.7,9,4,"{""bars"": 146, ""beat_duration_ms"": 667, ""beats""...",2026-06-13 19:18:35
6,7208611,1h6kgem1ai8vUgO1rZOwfB,131.995,132.00 BPM,allegro,646,4,455,162,4/4,...,3.6,7.3,14.5,29.1,58.2,116.4,10,5,"{""bars"": 162, ""beat_duration_ms"": 455, ""beats""...",2026-06-13 19:18:35
7,10261590,3Fc7k96EGOGiJBMZUxbpq7,120.461,120.00 BPM,allegro,464,4,498,116,4/4,...,4.0,8.0,15.9,31.9,63.8,127.5,7,3,"{""bars"": 116, ""beat_duration_ms"": 498, ""beats""...",2026-06-13 19:18:35
8,13594113,03tqyYWC9Um2ZqU0ZN849H,131.497,131.00 BPM,allegro,578,4,456,145,4/4,...,3.7,7.3,14.6,29.2,58.4,116.8,9,4,"{""bars"": 145, ""beat_duration_ms"": 456, ""beats""...",2026-06-13 19:18:35
9,14877728,3sh2q76qsc7yLkQNmHKfQf,130.024,130.00 BPM,allegro,409,4,461,102,4/4,...,3.7,7.4,14.8,29.5,59.1,118.1,6,3,"{""bars"": 102, ""beat_duration_ms"": 461, ""beats""...",2026-06-13 19:18:35



harmony


,rekordbox_track_id,spotify_track_id,key,mode,camelot,camelot_number,camelot_letter,note,raw_payload,fetched_at
0,257614,4zlbKky2yA657Sk5rekZoR,2,major,10B,10,B,D,"{""camelot"": ""10B"", ""camelot_letter"": ""B"", ""cam...",2026-06-13 19:18:34
1,549049,2x2iWUo1PYncvoUQO8M94J,6,major,2B,2,B,F#,"{""camelot"": ""2B"", ""camelot_letter"": ""B"", ""came...",2026-06-13 19:18:34
2,786655,1wecjDqjR28ENc6G4oGIBh,2,major,10B,10,B,D,"{""camelot"": ""10B"", ""camelot_letter"": ""B"", ""cam...",2026-06-13 19:18:34
3,1112161,54hA0ldJYyT1huGfSeOjdQ,7,major,9B,9,B,G,"{""camelot"": ""9B"", ""camelot_letter"": ""B"", ""came...",2026-06-13 19:18:34
4,1500026,29tRu5BqKiUDAoWLKIO3RN,5,minor,4A,4,A,Fm,"{""camelot"": ""4A"", ""camelot_letter"": ""A"", ""came...",2026-06-13 19:18:34
5,2967406,0h7JPd0usccTo7EoCXiNck,10,minor,3A,3,A,Bbm,"{""camelot"": ""3A"", ""camelot_letter"": ""A"", ""came...",2026-06-13 19:18:35
6,7208611,1h6kgem1ai8vUgO1rZOwfB,2,major,10B,10,B,D,"{""camelot"": ""10B"", ""camelot_letter"": ""B"", ""cam...",2026-06-13 19:18:35
7,10261590,3Fc7k96EGOGiJBMZUxbpq7,1,major,3B,3,B,Db,"{""camelot"": ""3B"", ""camelot_letter"": ""B"", ""came...",2026-06-13 19:18:35
8,13594113,03tqyYWC9Um2ZqU0ZN849H,1,major,3B,3,B,Db,"{""camelot"": ""3B"", ""camelot_letter"": ""B"", ""came...",2026-06-13 19:18:35
9,14877728,3sh2q76qsc7yLkQNmHKfQf,9,major,11B,11,B,A,"{""camelot"": ""11B"", ""camelot_letter"": ""B"", ""cam...",2026-06-13 19:18:35



score


,rekordbox_track_id,spotify_track_id,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,dance_floor,chill,aggressive,hype,groove,warmup,peak_time,blendability,vocal_risk,raw_payload,fetched_at
0,257614,4zlbKky2yA657Sk5rekZoR,0.847,0.724,0.047,0.004,0.003,0.073,0.326,0.792,0.001,0.721,0.833,0.879,0.300,0.761,0.504,0.427,"{""acousticness"": 0.004, ""aggressive"": 0.721, ""...",2026-06-13 19:18:34
1,549049,2x2iWUo1PYncvoUQO8M94J,0.736,0.792,0.146,0.062,0.000,0.223,0.699,0.761,0.013,0.743,0.860,0.771,0.305,0.775,0.446,0.488,"{""acousticness"": 0.062, ""aggressive"": 0.743, ""...",2026-06-13 19:18:34
2,786655,1wecjDqjR28ENc6G4oGIBh,0.592,0.986,0.105,0.040,0.000,0.151,0.182,0.769,0.001,0.946,0.981,0.683,0.218,0.868,0.432,0.463,"{""acousticness"": 0.04, ""aggressive"": 0.946, ""b...",2026-06-13 19:18:34
3,1112161,54hA0ldJYyT1huGfSeOjdQ,0.771,0.729,0.041,0.002,0.672,0.217,0.630,0.752,0.000,0.728,0.736,0.827,0.301,0.742,0.792,0.156,"{""acousticness"": 0.002, ""aggressive"": 0.728, ""...",2026-06-13 19:18:34
4,1500026,29tRu5BqKiUDAoWLKIO3RN,0.697,0.633,0.100,0.002,0.200,0.109,0.369,0.668,0.001,0.632,0.749,0.758,0.348,0.652,0.544,0.380,"{""acousticness"": 0.002, ""aggressive"": 0.632, ""...",2026-06-13 19:18:34
5,2967406,0h7JPd0usccTo7EoCXiNck,0.815,0.680,0.085,0.068,0.012,0.117,0.804,0.754,0.022,0.634,0.789,0.845,0.348,0.720,0.489,0.446,"{""acousticness"": 0.068, ""aggressive"": 0.634, ""...",2026-06-13 19:18:35
6,7208611,1h6kgem1ai8vUgO1rZOwfB,0.742,0.844,0.048,0.001,0.000,0.066,0.416,0.788,0.000,0.843,0.906,0.805,0.255,0.813,0.482,0.429,"{""acousticness"": 0.001, ""aggressive"": 0.843, ""...",2026-06-13 19:18:35
7,10261590,3Fc7k96EGOGiJBMZUxbpq7,0.884,0.549,0.471,0.035,0.070,0.112,0.394,0.733,0.016,0.530,0.710,0.778,0.376,0.649,0.393,0.655,"{""acousticness"": 0.035, ""aggressive"": 0.53, ""b...",2026-06-13 19:18:35
8,13594113,03tqyYWC9Um2ZqU0ZN849H,0.760,0.595,0.039,0.005,0.000,0.241,0.361,0.686,0.002,0.592,0.756,0.820,0.363,0.644,0.488,0.423,"{""acousticness"": 0.005, ""aggressive"": 0.592, ""...",2026-06-13 19:18:35
9,14877728,3sh2q76qsc7yLkQNmHKfQf,0.916,0.527,0.120,0.161,0.000,0.259,0.740,0.741,0.076,0.442,0.676,0.905,0.446,0.644,0.491,0.472,"{""acousticness"": 0.161, ""aggressive"": 0.442, ""...",2026-06-13 19:18:35



genres


,rekordbox_track_id,spotify_track_id,values,raw_payload,fetched_at
0,257614,4zlbKky2yA657Sk5rekZoR,"[""stutter house"", ""house"", ""dubstep"", ""edm"", ""...","[""stutter house"", ""house"", ""dubstep"", ""edm"", ""...",2026-06-13 19:18:34
1,549049,2x2iWUo1PYncvoUQO8M94J,"[""rap"", ""hip hop""]","[""rap"", ""hip hop""]",2026-06-13 19:18:34
2,786655,1wecjDqjR28ENc6G4oGIBh,"[""bass house"", ""edm"", ""bassline"", ""edm trap""]","[""bass house"", ""edm"", ""bassline"", ""edm trap""]",2026-06-13 19:18:34
3,1112161,54hA0ldJYyT1huGfSeOjdQ,"[""moombahton"", ""tech house"", ""edm""]","[""moombahton"", ""tech house"", ""edm""]",2026-06-13 19:18:34
4,1500026,29tRu5BqKiUDAoWLKIO3RN,"[""dubstep"", ""edm"", ""electro"", ""electronic"", ""d...","[""dubstep"", ""edm"", ""electro"", ""electronic"", ""d...",2026-06-13 19:18:34
5,2967406,0h7JPd0usccTo7EoCXiNck,"[""electro swing"", ""trip hop"", ""nu jazz"", ""down...","[""electro swing"", ""trip hop"", ""nu jazz"", ""down...",2026-06-13 19:18:35
6,7208611,1h6kgem1ai8vUgO1rZOwfB,"[""drill""]","[""drill""]",2026-06-13 19:18:35
7,10261590,3Fc7k96EGOGiJBMZUxbpq7,[],[],2026-06-13 19:18:35
8,13594113,03tqyYWC9Um2ZqU0ZN849H,[],[],2026-06-13 19:18:35
9,14877728,3sh2q76qsc7yLkQNmHKfQf,"[""tech house"", ""house"", ""latin house"", ""techno""]","[""tech house"", ""house"", ""latin house"", ""techno""]",2026-06-13 19:18:35


## Build Rekordbox Spotify Match Base

Create a joined analysis base with every Rekordbox track, its accepted Spotify match when present, and the derived match score.

In [7]:
if table_exists("spotify_tracks") and table_exists("rekordbox_spotify_matches"):
    track_columns = table_columns("rekordbox_tracks")
    match_columns = table_columns("rekordbox_spotify_matches")
    search_query_select = (
        "m.spotify_search_query_string"
        if "spotify_search_query_string" in match_columns
        else "rb.spotify_search_query_string"
        if "spotify_search_query_string" in track_columns
        else "NULL AS spotify_search_query_string"
    )
    rekordbox_spotify_match_base = pd.read_sql_query(
        """
        SELECT
            rb.rekordbox_track_id,
            rb.title AS rekordbox_title,
            rb.artist AS rekordbox_artist,
            rb.album AS rekordbox_album,
            rb.genre,
            rb.bpm,
            rb.key,
            rb.duration AS rekordbox_duration_seconds,
            rb.playlist_name,
            m.spotify_track_id,
            m.match_score,
            m.matched_at,
            {search_query_select},
            st.title AS spotify_title,
            st.artist_names AS spotify_artist_names,
            st.album_name AS spotify_album_name,
            st.duration_ms AS spotify_duration_ms,
            st.explicit AS spotify_explicit,
            st.spotify_url,
            CASE
                WHEN m.spotify_track_id IS NULL THEN 'unmatched'
                ELSE 'matched'
            END AS spotify_match_status
        FROM rekordbox_tracks rb
        LEFT JOIN rekordbox_spotify_matches m
            ON rb.rekordbox_track_id = m.rekordbox_track_id
        LEFT JOIN spotify_tracks st
            ON m.spotify_track_id = st.spotify_track_id
        ORDER BY rb.rekordbox_track_id
        """.format(search_query_select=search_query_select),
        conn,
    )
else:
    rekordbox_spotify_match_base = tracks.copy()
    if "spotify_search_query_string" not in rekordbox_spotify_match_base.columns:
        rekordbox_spotify_match_base["spotify_search_query_string"] = None
    rekordbox_spotify_match_base["spotify_match_status"] = "spotify tables missing"

print(f"Loaded {len(rekordbox_spotify_match_base)} rows in match base")
rekordbox_spotify_match_base.head(20)

Loaded 289 rows in match base


,rekordbox_track_id,rekordbox_title,rekordbox_artist,rekordbox_album,genre,bpm,key,rekordbox_duration_seconds,playlist_name,spotify_track_id,match_score,matched_at,spotify_search_query_string,spotify_title,spotify_artist_names,spotify_album_name,spotify_duration_ms,spotify_explicit,spotify_url,spotify_match_status
0,257614,baby again.. (original mix),"four tet, skrillex, fred again..",Baby again..,Bass House,127.00,Gm,319.0,Z,4zlbKky2yA657Sk5rekZoR,0.8333,2026-06-13 04:34:56,track: baby again artist: four tet album: Baby...,Baby again..,"Fred again.., Skrillex, Four Tet",Baby again..,319963.0,0.0,https://open.spotify.com/track/4zlbKky2yA657Sk...,matched
1,549049,go dj [explicit],lil wayne,Tha Carter [Explicit],Rap & Hip-Hop,79.00,Em,281.0,Z,2x2iWUo1PYncvoUQO8M94J,0.9690,2026-06-13 05:15:06,track: go dj artist: lil wayne album: Tha Cart...,Go DJ,Lil Wayne,Tha Carter,281826.0,1.0,https://open.spotify.com/track/2x2iWUo1PYncvoU...,matched
2,786655,murdah (isoxo remix),knock2,niteharts,Trap / Future Bass,145.00,Dm,172.0,Z,1wecjDqjR28ENc6G4oGIBh,0.9000,2026-06-13 05:15:08,track: murdah (isoxo remix) artist: knock2 alb...,murdah (ISOxo Remix),"Knock2, ISOxo",niteharts,172137.0,0.0,https://open.spotify.com/track/1wecjDqjR28ENc6...,matched
3,1112161,on my mind (original mix),"diplo, sidepiece",On My Mind,Bass House,123.00,Gm,189.0,Z,54hA0ldJYyT1huGfSeOjdQ,0.7682,2026-06-13 04:34:59,track: on my mind artist: diplo album: On My Mind,On My Mind,"Diplo, SIDEPIECE",Do You Dance?,189268.0,0.0,https://open.spotify.com/track/54hA0ldJYyT1huG...,matched
4,1500026,tears,"skrillex, joker, & sleepnet",,,140.00,Fm,185.0,Z,29tRu5BqKiUDAoWLKIO3RN,0.8387,2026-06-13 05:15:09,track: tears artist: skrillex album:,Tears,"Skrillex, Joker, Sleepnet",Quest For Fire,185571.0,0.0,https://open.spotify.com/track/29tRu5BqKiUDAoW...,matched
5,1516396,"madonna,the weeknd, playboi carti - popular [w...",NaN,,,99.00,Bbm,216.0,Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
6,2967406,just jammin' (original mix),gramatik,Coffee Shop Selection,Electronica,90.00,Ebm,389.0,Z,0h7JPd0usccTo7EoCXiNck,0.9083,2026-06-13 05:15:11,track: just jammin artist: gramatik album: Cof...,Just Jammin',Gramatik,SB2,389500.0,0.0,https://open.spotify.com/track/0h7JPd0usccTo7E...,matched
7,7208611,i don't like,chief keef,Still Rich,Drill Rap,131.99,F#m,294.0,Z,1h6kgem1ai8vUgO1rZOwfB,0.8727,2026-06-13 05:15:13,track: i don t like artist: chief keef album: ...,I Don't Like,"Chief Keef, Lil Reese",Finally Rich,293840.0,1.0,https://open.spotify.com/track/1h6kgem1ai8vUgO...,matched
8,8784213,popular (sped up),NaN,,,114.84,Abm,187.0,Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
9,9901726,skank n flex (original mix),"wax motif, scrufizzer, taiki nulight",House of Wax,Bass House,130.00,Fm,266.0,Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched


## Build DJ Audio Analysis Base

Create a joined analysis base with one row per enriched Rekordbox/Spotify match and aliased columns for each DJ analysis category.

In [8]:
def prefixed_select_columns(table_name, alias, prefix, exclude=("rekordbox_track_id", "spotify_track_id")):
    return [
        f'{alias}."{column}" AS {prefix}_{column}'
        for column in table_columns(table_name)
        if column not in exclude
    ]

required_dj_tables = set(dj_audio_analysis_table_names)
if required_dj_tables.issubset(set(table_names)):
    dj_select_columns = [
        "m.rekordbox_track_id",
        "m.spotify_track_id",
        "rb.title AS rekordbox_title",
        "rb.artist AS rekordbox_artist",
        "rb.album AS rekordbox_album",
        "st.title AS spotify_title",
        "st.artist_names AS spotify_artist_names",
        "st.album_name AS spotify_album_name",
    ]
    dj_select_columns.extend(prefixed_select_columns("track_analysis", "ta", "track_analysis"))
    dj_select_columns.extend(prefixed_select_columns("rhythm", "r", "rhythm"))
    dj_select_columns.extend(prefixed_select_columns("harmony", "h", "harmony"))
    dj_select_columns.extend(prefixed_select_columns("score", "s", "score"))
    dj_select_columns.extend(prefixed_select_columns("genres", "g", "genres"))

    dj_query = f"""
        SELECT
            {", ".join(dj_select_columns)}
        FROM rekordbox_spotify_matches m
        LEFT JOIN rekordbox_tracks rb
            ON m.rekordbox_track_id = rb.rekordbox_track_id
        LEFT JOIN spotify_tracks st
            ON m.spotify_track_id = st.spotify_track_id
        LEFT JOIN track_analysis ta
            ON m.rekordbox_track_id = ta.rekordbox_track_id
        LEFT JOIN rhythm r
            ON m.rekordbox_track_id = r.rekordbox_track_id
        LEFT JOIN harmony h
            ON m.rekordbox_track_id = h.rekordbox_track_id
        LEFT JOIN score s
            ON m.rekordbox_track_id = s.rekordbox_track_id
        LEFT JOIN genres g
            ON m.rekordbox_track_id = g.rekordbox_track_id
        WHERE ta.rekordbox_track_id IS NOT NULL
        ORDER BY m.rekordbox_track_id
    """
    dj_audio_analysis_base = pd.read_sql_query(dj_query, conn)
else:
    dj_audio_analysis_base = pd.DataFrame()

print(f"Loaded {len(dj_audio_analysis_base)} rows in DJ audio analysis base")
dj_audio_analysis_base.head(20)


Loaded 209 rows in DJ audio analysis base


,rekordbox_track_id,spotify_track_id,rekordbox_title,rekordbox_artist,rekordbox_album,spotify_title,spotify_artist_names,spotify_album_name,track_analysis_ids_isrc,track_analysis_duration_ms,...,score_groove,score_chill,score_speechiness,score_energy,score_vocal_risk,score_danceability,score_fetched_at,genres_values,genres_fetched_at,genres_raw_payload
0,257614,4zlbKky2yA657Sk5rekZoR,baby again.. (original mix),"four tet, skrillex, fred again..",Baby again..,Baby again..,"Fred again.., Skrillex, Four Tet",Baby again..,GBAHS2300215,319964,...,0.879,0.001,0.047,0.724,0.427,0.847,2026-06-13 19:18:34,"[""stutter house"", ""house"", ""dubstep"", ""edm"", ""...",2026-06-13 19:18:34,"[""stutter house"", ""house"", ""dubstep"", ""edm"", ""..."
1,549049,2x2iWUo1PYncvoUQO8M94J,go dj [explicit],lil wayne,Tha Carter [Explicit],Go DJ,Lil Wayne,Tha Carter,USCM50400067,281827,...,0.771,0.013,0.146,0.792,0.488,0.736,2026-06-13 19:18:34,"[""rap"", ""hip hop""]",2026-06-13 19:18:34,"[""rap"", ""hip hop""]"
2,786655,1wecjDqjR28ENc6G4oGIBh,murdah (isoxo remix),knock2,niteharts,murdah (ISOxo Remix),"Knock2, ISOxo",niteharts,QMUY42200172,172138,...,0.683,0.001,0.105,0.986,0.463,0.592,2026-06-13 19:18:34,"[""bass house"", ""edm"", ""bassline"", ""edm trap""]",2026-06-13 19:18:34,"[""bass house"", ""edm"", ""bassline"", ""edm trap""]"
3,1112161,54hA0ldJYyT1huGfSeOjdQ,on my mind (original mix),"diplo, sidepiece",On My Mind,On My Mind,"Diplo, SIDEPIECE",Do You Dance?,USZ4V1900134,189268,...,0.827,0.000,0.041,0.729,0.156,0.771,2026-06-13 19:18:34,"[""moombahton"", ""tech house"", ""edm""]",2026-06-13 19:18:34,"[""moombahton"", ""tech house"", ""edm""]"
4,1500026,29tRu5BqKiUDAoWLKIO3RN,tears,"skrillex, joker, & sleepnet",,Tears,"Skrillex, Joker, Sleepnet",Quest For Fire,USAT22300825,185571,...,0.758,0.001,0.100,0.633,0.380,0.697,2026-06-13 19:18:34,"[""dubstep"", ""edm"", ""electro"", ""electronic"", ""d...",2026-06-13 19:18:34,"[""dubstep"", ""edm"", ""electro"", ""electronic"", ""d..."
5,2967406,0h7JPd0usccTo7EoCXiNck,just jammin' (original mix),gramatik,Coffee Shop Selection,Just Jammin',Gramatik,SB2,USQY51422103,389500,...,0.845,0.022,0.085,0.680,0.446,0.815,2026-06-13 19:18:35,"[""electro swing"", ""trip hop"", ""nu jazz"", ""down...",2026-06-13 19:18:35,"[""electro swing"", ""trip hop"", ""nu jazz"", ""down..."
6,7208611,1h6kgem1ai8vUgO1rZOwfB,i don't like,chief keef,Still Rich,I Don't Like,"Chief Keef, Lil Reese",Finally Rich,USUM71207543,293840,...,0.805,0.000,0.048,0.844,0.429,0.742,2026-06-13 19:18:35,"[""drill""]",2026-06-13 19:18:35,"[""drill""]"
7,10261590,3Fc7k96EGOGiJBMZUxbpq7,lite spots (original mix),kaytranada,99.9%%,LITE SPOTS,KAYTRANADA,99.9%,GBBKS1600025,230920,...,0.778,0.016,0.471,0.549,0.655,0.884,2026-06-13 19:18:35,[],2026-06-13 19:18:35,[]
8,13594113,03tqyYWC9Um2ZqU0ZN849H,no hands (feat. roscoe dash and wale) [amended...,waka flocka flame,Flockaveli [Clean],No Hands (feat. Roscoe Dash & Wale),"Waka Flocka Flame, Roscoe Dash, Wale",Flockaveli,USWB11002103,263773,...,0.820,0.002,0.039,0.595,0.423,0.760,2026-06-13 19:18:35,[],2026-06-13 19:18:35,[]
9,14877728,3sh2q76qsc7yLkQNmHKfQf,stephanie (original mix),"cloonee, young m.a, inntraw",Stephanie,Stephanie,"Cloonee, Young M.A, InntRaw",Stephanie,GBLV62409221,188534,...,0.905,0.076,0.120,0.527,0.472,0.916,2026-06-13 19:18:35,"[""tech house"", ""house"", ""latin house"", ""techno""]",2026-06-13 19:18:35,"[""tech house"", ""house"", ""latin house"", ""techno""]"


## Quick Manual Analysis

In [9]:
summary = {
    "tracks": len(tracks),
    "artists": tracks["artist"].nunique(dropna=True) if "artist" in tracks else None,
    "genres": tracks["genre"].nunique(dropna=True) if "genre" in tracks else None,
    "playlists": tracks["playlist_name"].nunique(dropna=True) if "playlist_name" in tracks else None,
    "avg_bpm": tracks["bpm"].mean() if "bpm" in tracks else None,
    "spotify_tracks": len(spotify_tracks),
    "spotify_matches": len(rekordbox_spotify_matches),
    "spotify_match_rate": (
        len(rekordbox_spotify_matches) / len(tracks)
        if len(tracks) else None
    ),
    "dj_track_analysis": len(track_analysis),
    "dj_rhythm": len(rhythm),
    "dj_harmony": len(harmony),
    "dj_score": len(score),
    "dj_genres": len(genres),
    "dj_audio_analysis_rate": (
        len(track_analysis) / len(rekordbox_spotify_matches)
        if len(rekordbox_spotify_matches) else None
    ),
}
summary

{'tracks': 289,
 'artists': 201,
 'genres': 35,
 'playlists': 1,
 'avg_bpm': np.float64(124.8187543252595),
 'spotify_tracks': 276,
 'spotify_matches': 209,
 'spotify_match_rate': 0.7231833910034602,
 'dj_track_analysis': 209,
 'dj_rhythm': 209,
 'dj_harmony': 209,
 'dj_score': 209,
 'dj_genres': 209,
 'dj_audio_analysis_rate': 1.0}

## Spotify Match Analysis

In [10]:
if "spotify_match_status" in rekordbox_spotify_match_base:
    display(
        rekordbox_spotify_match_base["spotify_match_status"]
        .value_counts(dropna=False)
        .rename_axis("spotify_match_status")
        .reset_index(name="tracks")
    )

if "match_score" in rekordbox_spotify_match_base and rekordbox_spotify_match_base["match_score"].notna().any():
    display(rekordbox_spotify_match_base["match_score"].describe())
    display(
        rekordbox_spotify_match_base
        .dropna(subset=["match_score"])
        .sort_values("match_score")
        [[
            "rekordbox_track_id",
            "rekordbox_title",
            "rekordbox_artist",
            "spotify_track_id",
            "spotify_title",
            "spotify_artist_names",
            "match_score",
            "spotify_search_query_string",
            "spotify_url",
        ]]
        .head(20)
    )
else:
    print("No Spotify match scores available yet. Run scripts/enrich_spotify.py to populate matches.")

,spotify_match_status,tracks
0,matched,209
1,unmatched,80


count    209.000000
mean       0.915480
std        0.078269
min        0.750000
25%        0.849000
50%        0.923000
75%        1.000000
max        1.000000
Name: match_score, dtype: float64

,rekordbox_track_id,rekordbox_title,rekordbox_artist,spotify_track_id,spotify_title,spotify_artist_names,match_score,spotify_search_query_string,spotify_url
147,129316216,paris (extended mix),tom enzy,2H7F7EfsVhy0jNLsVz8MLH,Paris,Else,0.7500,track: paris artist: tom enzy album: Paris,https://open.spotify.com/track/2H7F7EfsVhy0jNL...
248,231210264,ocho (beauz & kevu vip) [extended mix] (beauz ...,"kevu, beauz",3vQuejTzsv7cMMa8Mo2QvB,Ocho,BEAUZ,0.7500,track: ocho artist: kevu album: Ocho (BEAUZ & ...,https://open.spotify.com/track/3vQuejTzsv7cMMa...
208,192874243,el baile,heynegaard & lvga,54x5G6IhYHNfUT8l6MM9GR,El Baile,"HP. Hoeger, M. Lackmaier",0.7525,track: el baile artist: heynegaard album: El B...,https://open.spotify.com/track/54x5G6IhYHNfUT8...
199,185686767,switch,"coco palmer, deadfreshfruit",6EjXU1XgECsr6LGZqrYKpp,SWITCH,Eatin' & Zachor,0.7536,track: switch artist: coco palmer album:,https://open.spotify.com/track/6EjXU1XgECsr6LG...
136,122914465,rhyme dust (extended),"mk, dom dolla",5mKiwDDrwG22qKKVL6JZqF,Rhyme Dust,"MK, Dom Dolla",0.7547,track: rhyme dust artist: mk album: Rhyme Dust...,https://open.spotify.com/track/5mKiwDDrwG22qKK...
263,242481537,push it (eastmix),dave east,4eswve9pQufeesxrM5wBxj,Push It - EASTMIX,Dave East,0.7576,track: push it artist: dave east album:,https://open.spotify.com/track/4eswve9pQufeesx...
253,236413131,in da getto (chris lorenzo remix),"skrillex, j. balvin",3O9BNmxmQ5M02TFGMWptJt,In Da Getto - Chris Lorenzo Remix,"J Balvin, Skrillex, Chris Lorenzo",0.7631,track: in da getto (chris lorenzo remix) artis...,https://open.spotify.com/track/3O9BNmxmQ5M02TF...
3,1112161,on my mind (original mix),"diplo, sidepiece",54hA0ldJYyT1huGfSeOjdQ,On My Mind,"Diplo, SIDEPIECE",0.7682,track: on my mind artist: diplo album: On My Mind,https://open.spotify.com/track/54hA0ldJYyT1huG...
202,189993643,move all night (extended mix),"kole, autograf, snbrn",6kNx6oySNfaqH3P4CGiKVL,Move All Night (feat. Kole),"SNBRN, Autograf, KOLE",0.7726,track: move all night artist: kole album: Move...,https://open.spotify.com/track/6kNx6oySNfaqH3P...
31,28626495,deposits (eastmix),dave east,2ByNcK3MKHJmKwryTY7oAL,Deposits - EASTMIX,Dave East,0.7778,track: deposits artist: dave east album:,https://open.spotify.com/track/2ByNcK3MKHJmKwr...


## DJ Audio Analysis Summary

In [14]:
if dj_audio_analysis_base.empty:
    print("No DJ audio analysis rows available yet. Run scripts/enrich_dj_audio_analysis.py to populate analysis tables.")
else:
    summary_columns = [
        "rhythm_bpm",
        "score_danceability",
        "score_energy",
        "score_dance_floor",
        "score_hype",
        "score_groove",
        "score_blendability",
        "score_vocal_risk",
    ]
    existing_summary_columns = [
        column for column in summary_columns
        if column in dj_audio_analysis_base.columns
    ]
    if existing_summary_columns:
        display(dj_audio_analysis_base[existing_summary_columns].describe())

    note_column = (
        "harmony_note"
        if "harmony_note" in dj_audio_analysis_base.columns
        else "note"
        if "note" in dj_audio_analysis_base.columns
        else None
    )
    if note_column:
        display(
            dj_audio_analysis_base[note_column]
            .value_counts(dropna=False)
            .rename_axis("note")
            .reset_index(name="tracks")
        )

    preview_columns = [
        "rekordbox_track_id",
        "rekordbox_title",
        "spotify_track_id",
        "track_analysis_name",
        "rhythm_bpm",
        "harmony_note",
        "score_dance_floor",
        "score_blendability",
        "genres_values",
    ]
    display(
        dj_audio_analysis_base[
            [column for column in preview_columns if column in dj_audio_analysis_base.columns]
        ].head(20)
    )


,rhythm_bpm,score_danceability,score_energy,score_dance_floor,score_hype,score_groove,score_blendability,score_vocal_risk
count,209.000000,209.000000,209.000000,209.000000,209.000000,209.000000,209.000000,209.000000
mean,127.703120,0.723847,0.792732,0.754828,0.818349,0.777512,0.587550,0.345584
std,19.540062,0.108424,0.163760,0.085478,0.115468,0.077511,0.160292,0.152529
min,74.981000,0.460000,0.334000,0.456000,0.350000,0.559000,0.321000,0.052000
25%,124.002000,0.646000,0.687000,0.697000,0.746000,0.731000,0.468000,0.230000
50%,126.074000,0.734000,0.827000,0.766000,0.837000,0.780000,0.515000,0.402000
75%,137.998000,0.803000,0.935000,0.810000,0.908000,0.836000,0.743000,0.439000
max,180.034000,0.978000,1.000000,0.933000,0.994000,0.958000,0.933000,0.658000


,note,tracks
0,Db,27
1,D,18
2,Bbm,17
3,F#m,16
4,Bm,15
5,G,12
6,Dbm,11
7,A,10
8,B,10
9,Abm,9


,rekordbox_track_id,rekordbox_title,spotify_track_id,track_analysis_name,rhythm_bpm,harmony_note,score_dance_floor,score_blendability,genres_values
0,257614,baby again.. (original mix),4zlbKky2yA657Sk5rekZoR,Baby again..,126.983,D,0.792,0.504,"[""stutter house"", ""house"", ""dubstep"", ""edm"", ""..."
1,549049,go dj [explicit],2x2iWUo1PYncvoUQO8M94J,Go DJ,157.955,F#,0.761,0.446,"[""rap"", ""hip hop""]"
2,786655,murdah (isoxo remix),1wecjDqjR28ENc6G4oGIBh,murdah (ISOxo Remix),144.937,D,0.769,0.432,"[""bass house"", ""edm"", ""bassline"", ""edm trap""]"
3,1112161,on my mind (original mix),54hA0ldJYyT1huGfSeOjdQ,On My Mind,123.027,G,0.752,0.792,"[""moombahton"", ""tech house"", ""edm""]"
4,1500026,tears,29tRu5BqKiUDAoWLKIO3RN,Tears,140.216,Fm,0.668,0.544,"[""dubstep"", ""edm"", ""electro"", ""electronic"", ""d..."
5,2967406,just jammin' (original mix),0h7JPd0usccTo7EoCXiNck,Just Jammin',90.008,Bbm,0.754,0.489,"[""electro swing"", ""trip hop"", ""nu jazz"", ""down..."
6,7208611,i don't like,1h6kgem1ai8vUgO1rZOwfB,I Don't Like,131.995,D,0.788,0.482,"[""drill""]"
7,10261590,lite spots (original mix),3Fc7k96EGOGiJBMZUxbpq7,LITE SPOTS,120.461,Db,0.733,0.393,[]
8,13594113,no hands (feat. roscoe dash and wale) [amended...,03tqyYWC9Um2ZqU0ZN849H,No Hands (feat. Roscoe Dash & Wale),131.497,Db,0.686,0.488,[]
9,14877728,stephanie (original mix),3sh2q76qsc7yLkQNmHKfQf,Stephanie,130.024,A,0.741,0.491,"[""tech house"", ""house"", ""latin house"", ""techno""]"


In [12]:
if "playlist_name" in tracks:
    display(tracks["playlist_name"].value_counts().rename_axis("playlist_name").reset_index(name="tracks"))

if "genre" in tracks:
    display(tracks["genre"].value_counts(dropna=False).head(20).rename_axis("genre").reset_index(name="tracks"))

if "artist" in tracks:
    display(tracks["artist"].value_counts(dropna=False).head(20).rename_axis("artist").reset_index(name="tracks"))

,playlist_name,tracks
0,Z,289


,genre,tracks
0,,50
1,Tech House,37
2,Bass House,29
3,Dance / Electro Pop,28
4,House,17
5,Electronica,14
6,Dance / Pop,13
7,Rap & Hip-Hop,9
8,Rap,8
9,Trap / Future Bass,7


,artist,tracks
0,NaN,30
1,knock2,12
2,isoxo,7
3,boris brejcha,5
4,gramatik,4
5,mau p,4
6,funk tribu,4
7,nicolas julian,3
8,rezz,3
9,biscits,3


In [13]:
unmatched_columns = [
    "rekordbox_track_id",
    "rekordbox_title",
    "rekordbox_artist",
    "rekordbox_album",
    "spotify_search_query_string",
    "spotify_match_status",
]
rekordbox_spotify_match_base.loc[
    rekordbox_spotify_match_base["spotify_match_status"] == "unmatched",
    [column for column in unmatched_columns if column in rekordbox_spotify_match_base.columns],
]

,rekordbox_track_id,rekordbox_title,rekordbox_artist,rekordbox_album,spotify_search_query_string,spotify_match_status
5,1516396,"madonna,the weeknd, playboi carti - popular [w...",NaN,,NaN,unmatched
8,8784213,popular (sped up),NaN,,NaN,unmatched
9,9901726,skank n flex (original mix),"wax motif, scrufizzer, taiki nulight",House of Wax,NaN,unmatched
11,10278033,new wave (extended mix),"andata, funk tribu",New Wave,NaN,unmatched
12,12870033,go crazy (redricepapi blend),redricepapi,another round,NaN,unmatched
...,...,...,...,...,...,...
266,245946343,coolio - gangsta's paradise (funk tribu edit),NaN,,NaN,unmatched
268,247028780,bumba bass (extended mix),"henry fong, tv noise",Bumba Bass - Extended Mix,NaN,unmatched
277,260432339,move shake drop,NaN,,NaN,unmatched
282,264032928,thrash (party starter) (party starter),"isoxo, knock2, isoknock",4EVR,NaN,unmatched
